# Swarm pipeline — Google Colab setup

Run the multi-agent stigmergic swarm on a Colab GPU. The pipeline auto-detects the GPU tier (T4 / L4 / A100-40 / A100-80) and picks a Qwen-Instruct model + dtype + agent populations to match. Single non-quantized model in VRAM; agents fire in parallel via vLLM's AsyncLLMEngine continuous batcher.

**What this notebook does:**
1. (Optional) Mount Google Drive for persistent outputs.
2. Configure path env vars.
3. Clone the repository.
4. Install vLLM + helper libs.
5. (Optional) Set `HF_TOKEN` for higher Hugging Face download rate limits.
6. Smoke-test with MOCK_LLM=1.
7. Run for real. The first run downloads the model (~14 GB for Qwen-7B at fp16).
8. Inspect outputs.

**Recommended runtime:** GPU → T4 (free tier) works for Qwen-7B-Instruct; L4 or A100 lets the auto-tier pick 14B or 32B.

## 1. Mount Google Drive (optional)

Skip this cell to keep everything ephemeral under `/content/`. Mount to persist runs to `/content/drive/MyDrive/swarm/` so subsequent sessions resume with model cache + outputs intact.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure paths

Flip `USE_DRIVE` to decide between Drive (persistent) and `/content/` (ephemeral). `HF_HOME` on Drive means the model download only runs once across sessions; it's the single biggest time saver on this notebook.

In [ ]:
import os

USE_DRIVE = True  # set False for fully ephemeral /content/

BASE = '/content/drive/MyDrive/swarm' if USE_DRIVE else '/content/swarm'

os.environ['SWARM_OUTPUTS_BASE_DIR']    = f'{BASE}/runs'
os.environ['SWARM_KB_DIR']              = f'{BASE}/knowledge_base'
os.environ['SWARM_RETRIEVAL_CACHE_DIR'] = f'{BASE}/retrieval_cache'
os.environ['HF_HOME']                   = f'{BASE}/hf_cache'
# Force the Colab path even if torch.cuda detection is flaky in the
# notebook kernel — the pipeline reads this to select vLLM.
os.environ['COLAB'] = '1'

for d in ('runs', 'knowledge_base', 'retrieval_cache', 'hf_cache'):
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

print('configured:')
for k in sorted(os.environ):
    if k.startswith('SWARM_') or k in ('COLAB', 'HF_HOME'):
        print(f'  {k} = {os.environ[k]}')

## 3. Clone the repository

The repo root *is* the `Attempt At Cleaning` directory (no nested subfolder).

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/sfuqua6/Stigmeric-Coordination.git'
REPO_DIR = '/content/swarm_repo'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    print(f'{REPO_DIR} exists; running git pull')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
print('cwd =', os.getcwd())
print('expected files:', sorted(f for f in os.listdir() if not f.startswith('.'))[:10])

## 4. Install dependencies

vLLM brings its own torch with CUDA support, so we don't need to install torch separately. The first install on Colab takes ~5 min; subsequent sessions on the same VM reuse the wheel cache.

In [ ]:
!pip install -q vllm
!pip install -q "bitsandbytes>=0.46.1"
!pip install -q sentence-transformers wikipedia beautifulsoup4 requests tqdm

## 5. (Optional) Hugging Face token

Unauthenticated downloads are rate-limited. Set `HF_TOKEN` if you have one — it makes the first-run model download faster and avoids the 'unauthenticated requests' warning. Get a token at huggingface.co/settings/tokens (Read access is sufficient).

In [ ]:
# Paste your token here, or leave blank to skip.
HF_TOKEN = ''
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF_TOKEN set')
else:
    print('no HF_TOKEN — downloads will be rate-limited (still works, just slower)')

In [ ]:
import subprocess, os

HF_CACHE = '/content/hf_cache'
os.makedirs(HF_CACHE, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE
os.environ['HF_HUB_CACHE'] = HF_CACHE
os.environ['TRANSFORMERS_CACHE'] = HF_CACHE

MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
print(f'Pre-warming HF cache: downloading {MODEL_NAME} to {HF_CACHE}...')
print('(First run: ~14 GiB, ~90s at 150 MB/s; subsequent runs load from cache in ~30s)')

try:
    subprocess.run([
        'huggingface-cli', 'download', MODEL_NAME, '--local-dir', 
        f'{HF_CACHE}/models--Qwen--Qwen2.5-7B-Instruct'
    ], check=True, timeout=600)
    print('✓ Model cached on local SSD')
except Exception as e:
    print(f'Warning: cache pre-warm failed ({e}), will download on first run')


## 4b. Pre-warm HuggingFace cache on local SSD

The Qwen2.5-7B model (~14 GiB) downloads to `/content/hf_cache/` on Colab's local SSD the first time, then subsequent runs load from cache in ~30 seconds. Downloading to Google Drive (FUSE.DRIVE) is 10-20x slower; the SSD is the right place.


## 6. Smoke test (MOCK_LLM=1, no model download)

Verifies the path wiring before you commit to the model download. Output should land under `$SWARM_OUTPUTS_BASE_DIR/outputs_mock/`.

In [ ]:
!MOCK_LLM=1 python run_swarm.py debate "Test thesis" --corpus=placeholder

## 7. Real run

First invocation downloads the model from Hugging Face (~14 GB for Qwen-7B-Instruct at fp16 on a T4; ~28 GB for Qwen-14B on L4). The pipeline auto-selects the model based on detected GPU tier; override with `SWARM_MODEL` if you want a specific one.

Don't pass `--heterogeneous` here — that's the laptop GGUF path. Colab uses a single model + vLLM internal batching. If you pass it anyway, the pipeline prints a warning and ignores it.

In [ ]:
# Override model if you want a specific one:
# os.environ['SWARM_MODEL'] = 'Qwen/Qwen2.5-3B-Instruct'   # smaller, faster
# os.environ['SWARM_MODEL'] = 'Qwen/Qwen2.5-7B-Instruct-AWQ'  # 4-bit AWQ if T4 OOMs at fp16
!python run_swarm.py debate "Does free will exist?"

## 8. Alternative — phase-isolated orchestrator

Spawns one subprocess per phase. Slower (~1–2 min subprocess overhead on Colab) but **crash-resumable**: if Colab disconnects mid-run, re-run the same command with the same `--run-id` and it skips completed phases.

Use this if you've hit OOM mid-round, or if your Colab session is unstable.

In [ ]:
!python tools/run_isolated.py debate "Does free will exist?" --run-id=free_will_isolated

## 9. Inspect outputs

Each run produces `answer.txt`, `signals.json`, `summary.json`, `round_log.json`, `citations.json`, `lineage.dot`, `run_meta.json`. The Colab migration added a few migration-relevant fields:

- `run_meta.json:colab_tier` — detected tier
- `run_meta.json:vllm_concurrency` — should be 32 on Colab
- `summary.json:wall_clock_s` — total time including model load + retrieval
- `summary.json:concurrent_calls_peak` — peak in-flight LLM calls
- `summary.json:tokens_per_second` — approximate throughput
- `round_log.json[i].diversity.cross_model_delta` — None on Colab (single model)

In [ ]:
import glob, json
from pathlib import Path

runs_dir = Path(os.environ['SWARM_OUTPUTS_BASE_DIR'], 'outputs')
runs = sorted(runs_dir.glob('*'), key=lambda p: p.stat().st_mtime) if runs_dir.exists() else []
if runs:
    latest = runs[-1]
    print('latest run:', latest)
    print()
    print('=== summary.json ===')
    print(json.dumps(json.loads((latest / 'summary.json').read_text()), indent=2))
    print()
    print('=== run_meta.json (migration fields) ===')
    m = json.loads((latest / 'run_meta.json').read_text())
    for k in ('colab_tier', 'vllm_concurrency', 'vllm_dtype', 'population_scaled_for', 'model_loaded_once', 'execution_mode'):
        if k in m:
            print(f'  {k}: {m[k]}')
    print()
    print('=== answer.txt ===')
    print((latest / 'answer.txt').read_text())
else:
    print('no real-LLM runs yet at', runs_dir)

## Troubleshooting

- **`CUDA out of memory` on T4 at fp16.** Qwen-7B is ~14 GB and T4 has 16 GB VRAM — close to the edge. Options: switch to an AWQ-quantized variant: `os.environ['SWARM_MODEL'] = 'Qwen/Qwen2.5-7B-Instruct-AWQ'`. Or pick a smaller model: `Qwen/Qwen2.5-3B-Instruct`.
- **First model download is slow.** Set `HF_TOKEN` (cell 5) for higher rate limits.
- **`tier=None` in run_meta.** GPU detection failed. The `COLAB=1` env var (cell 2) forces the tier-unknown path which still uses Qwen-7B-Instruct + vLLM.
- **Session disconnects mid-run.** Use cell 8 (phase-isolated). With `USE_DRIVE=True`, the `store_state.json` checkpoint is on Drive — re-run with the same `--run-id` and it resumes from the last completed phase.
- **`vllm install fails`.** Colab sometimes has stale CUDA toolkit. Try `!pip install -q --upgrade vllm`. As a last resort, the laptop GGUF path still works under `!SWARM_BACKEND=gguf python run_swarm.py ...` but you'd need to download GGUFs by hand.